In [1]:
# ==============================================================================
# INITIALISATION ET IMPORTATION DES LIBRAIRIES
# ==============================================================================
import os
import re
import string
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import nltk

# --- Outils de Traitement du Langage (NLP) et Similarité ---
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from wordcloud import WordCloud

# --- Machine Learning ---
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# --- Explainable AI ---
import shap

# --- Configuration visuelle et environnement ---
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
sns.set_theme(style="whitegrid")

# Téléchargement des dictionnaires NLTK
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('vader_lexicon', quiet=True)

print("✅ Cellule 1 terminée : Tous les outils sont chargés.")

✅ Cellule 1 terminée : Tous les outils sont chargés.


In [3]:
def auditer_et_corriger_dataset(dataframe):
    print("--- DÉBUT DE L'AUDIT DE QUALITÉ ---")
    df_propre = dataframe.copy()
    
    taille_avant = len(df_propre)
    df_propre = df_propre.dropna(subset=['text_', 'label'])
    if taille_avant - len(df_propre) > 0:
        print(f"🔧 CORRECTION : {taille_avant - len(df_propre)} lignes nulles supprimées.")

    taille_avant = len(df_propre)
    df_propre = df_propre[df_propre['label'].isin(['CG', 'OR'])]
    if taille_avant - len(df_propre) > 0:
        print(f"🔧 CORRECTION : {taille_avant - len(df_propre)} labels aberrants exclus.")

    taille_avant = len(df_propre)
    df_propre['text_'] = df_propre['text_'].replace(r'^\s*$', np.nan, regex=True)
    df_propre = df_propre.dropna(subset=['text_'])
    
    print(f"🟢 DATASET VALIDÉ : Taille finale = {len(df_propre)} lignes.")
    return df_propre

df = pd.read_csv('fake reviews dataset.csv')
df = auditer_et_corriger_dataset(df)
df = df.drop_duplicates().reset_index(drop=True)

--- DÉBUT DE L'AUDIT DE QUALITÉ ---
🟢 DATASET VALIDÉ : Taille finale = 40432 lignes.
